In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import glob
print(glob.glob("/content/drive/MyDrive/FinHOLLY/data/my_adapter_train.csv"))

In [ ]:
!pip install -q transformers peft accelerate sentencepiece pandas sacremoses sacrebleu
!pip install -q --upgrade torchao

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import torch

DRIVE_ROOT = "/content/drive/MyDrive/FinHOLLY"
DATA_DIR = f"{DRIVE_ROOT}/data"
OUT_DIR = f"{DRIVE_ROOT}/my_language_adapter_run"
os.makedirs(OUT_DIR, exist_ok=True)

TRAIN_CSV = f"{DATA_DIR}/my_adapter_train.csv"

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음(런타임을 GPU로 바꾸세요)")
if torch.cuda.is_available():
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")

In [ ]:
import pandas as pd

SRC_LANG = "kor_Hang"
TGT_LANG = "mya_Mymr"
VAL_RATIO = 0.1
SEED = 42

df = pd.read_csv(TRAIN_CSV, encoding="utf-8-sig")
df = df.dropna(subset=["ko", "my"]).reset_index(drop=True)
print(f"전체: {len(df)}쌍")

shuffled = df.sample(frac=1, random_state=SEED).reset_index(drop=True)
n_val = int(len(shuffled) * VAL_RATIO)
val_df = shuffled.iloc[:n_val].reset_index(drop=True)
train_df = shuffled.iloc[n_val:].reset_index(drop=True)

print(f"train: {len(train_df)}쌍, val: {len(val_df)}쌍")

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from peft import LoraConfig, get_peft_model, TaskType

BASE_MODEL = "facebook/nllb-200-distilled-1.3B"
MAX_LENGTH = 128

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
model = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL, torch_dtype=torch.float16)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    bias="none",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

In [ ]:
from torch.utils.data import Dataset, DataLoader

class TranslationPairDataset(Dataset):
    def __init__(self, df: pd.DataFrame):
        self.rows = df.reset_index(drop=True)

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        r = self.rows.iloc[idx]
        return {"ko": str(r["ko"]), "my": str(r["my"])}

def make_collate_fn(tokenizer, src_lang=SRC_LANG, tgt_lang=TGT_LANG, max_length=MAX_LENGTH):
    def collate(batch):
        tokenizer.src_lang = src_lang
        tokenizer.tgt_lang = tgt_lang
        sources = [b["ko"] for b in batch]
        targets = [b["my"] for b in batch]
        enc = tokenizer(
            sources, text_target=targets, return_tensors="pt",
            padding=True, truncation=True, max_length=max_length,
        )
        labels = enc["labels"].clone()
        labels[labels == tokenizer.pad_token_id] = -100
        return {
            "input_ids": enc["input_ids"],
            "attention_mask": enc["attention_mask"],
            "labels": labels,
        }

    return collate

BATCH_SIZE = 4
GRAD_ACCUM_STEPS = 8

train_ds = TranslationPairDataset(train_df)
val_ds = TranslationPairDataset(val_df)
collate_fn = make_collate_fn(tokenizer)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

print(f"배치 크기: {BATCH_SIZE}, epoch당 배치 수: {len(train_loader)}")

In [ ]:
from transformers import get_linear_schedule_with_warmup

EPOCHS = 2
LR = 5e-4
WARMUP_RATIO = 0.1

optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LR)
total_steps = (len(train_loader) // GRAD_ACCUM_STEPS) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=int(total_steps * WARMUP_RATIO), num_training_steps=total_steps
)
scaler = torch.cuda.amp.GradScaler()

model.train()
for epoch in range(EPOCHS):
    epoch_loss, n_batches = 0.0, 0
    optimizer.zero_grad()
    for step, batch in enumerate(train_loader):
        batch = {k: v.to(device) for k, v in batch.items()}
        with torch.cuda.amp.autocast(dtype=torch.float16):
            out = model(
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"],
                labels=batch["labels"],
            )
            loss = out.loss / GRAD_ACCUM_STEPS

        scaler.scale(loss).backward()
        epoch_loss += out.loss.item()
        n_batches += 1

        if (step + 1) % GRAD_ACCUM_STEPS == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()

        if (step + 1) % 500 == 0:
            print(f"  epoch {epoch+1} step {step+1}/{len(train_loader)}  CE={epoch_loss/n_batches:.4f}")

    print(f"[epoch {epoch+1}/{EPOCHS}] 평균 CE={epoch_loss/n_batches:.4f}")

    ckpt_dir = f"{OUT_DIR}/checkpoint-epoch{epoch+1}"
    model.save_pretrained(ckpt_dir)
    tokenizer.save_pretrained(ckpt_dir)
    print(f"  체크포인트 저장: {ckpt_dir}")

print("학습 완료")

In [ ]:
import sacrebleu

@torch.no_grad()
def translate_batch(model, ko_texts, use_adapter=True, batch_size=16):
    tokenizer.src_lang = SRC_LANG
    forced_bos = tokenizer.convert_tokens_to_ids(TGT_LANG)
    outputs = []
    for i in range(0, len(ko_texts), batch_size):
        batch = ko_texts[i : i + batch_size]
        inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=MAX_LENGTH).to(device)
        if use_adapter:
            out_ids = model.generate(**inputs, forced_bos_token_id=forced_bos, max_length=MAX_LENGTH, num_beams=4)
        else:
            with model.disable_adapter():
                out_ids = model.generate(**inputs, forced_bos_token_id=forced_bos, max_length=MAX_LENGTH, num_beams=4)
        outputs.extend(tokenizer.batch_decode(out_ids, skip_special_tokens=True))
    return outputs

model.eval()

EVAL_N = 500
eval_df = val_df.sample(min(EVAL_N, len(val_df)), random_state=7).reset_index(drop=True)
ko_texts = eval_df["ko"].tolist()
refs = eval_df["my"].tolist()

print(f"=== val {len(ko_texts)}개로 baseline(순정 NLLB) 평가 ===")
baseline_hyps = translate_batch(model, ko_texts, use_adapter=False)
baseline_chrf = sacrebleu.corpus_chrf(baseline_hyps, [refs])
print(f"baseline chrF: {baseline_chrf.score:.2f}")

print(f"\n=== val {len(ko_texts)}개로 언어 어댑터 평가 ===")
adapter_hyps = translate_batch(model, ko_texts, use_adapter=True)
adapter_chrf = sacrebleu.corpus_chrf(adapter_hyps, [refs])
print(f"adapter chrF: {adapter_chrf.score:.2f}")

print(f"\n=== Before vs After ===")
print(f"chrF: {baseline_chrf.score:.2f} -> {adapter_chrf.score:.2f}  ({adapter_chrf.score-baseline_chrf.score:+.2f})")

result_df = pd.DataFrame({
    "ko": ko_texts, "reference": refs,
    "baseline_mt": baseline_hyps, "adapter_mt": adapter_hyps,
})
result_df.to_csv(f"{OUT_DIR}/val_eval_result.csv", index=False, encoding="utf-8-sig")
print(f"\n상세 결과 저장: {OUT_DIR}/val_eval_result.csv")

In [ ]:
import pandas as pd
from sentence_transformers import SentenceTransformer
import numpy as np

DATA_DIR = "/content/drive/MyDrive/FinHOLLY/data"
OUT_DIR = "/content/drive/MyDrive/FinHOLLY/my_language_adapter_run"

df = pd.read_csv(f"{DATA_DIR}/my_adapter_train.csv", encoding="utf-8-sig")
df = df.dropna(subset=["ko", "my"]).reset_index(drop=True)
shuffled = df.sample(frac=1, random_state=42).reset_index(drop=True)
n_val = int(len(shuffled) * 0.1)
val_df = shuffled.iloc[:n_val].reset_index(drop=True)
print(f"원본 val: {len(val_df)}쌍")

print("LaBSE 로딩...")
labse = SentenceTransformer("sentence-transformers/LaBSE")

print("정렬 유사도 계산 중...")
ko_emb = labse.encode(val_df["ko"].tolist(), normalize_embeddings=True, show_progress_bar=True, batch_size=64)
my_emb = labse.encode(val_df["my"].tolist(), normalize_embeddings=True, show_progress_bar=True, batch_size=64)

sims = (ko_emb * my_emb).sum(axis=1)
val_df["align_sim"] = sims

print(f"\n=== 정렬 유사도 분포 ===")
print(f"평균: {sims.mean():.3f}, 중앙값: {np.median(sims):.3f}")
print(f"하위 10%: {np.percentile(sims,10):.3f}, 하위 30%: {np.percentile(sims,30):.3f}")
print(f"상위 30%: {np.percentile(sims,70):.3f}")

print(f"\n=== 유사도 낮은 것 (깨진 정렬 의심) 5개 ===")
worst = val_df.nsmallest(5, "align_sim")
for _, r in worst.iterrows():
    print(f"  sim={r['align_sim']:.3f} | ko: {r['ko'][:30]} | my: {r['my'][:30]}")

THRESHOLD = np.percentile(sims, 30)
clean = val_df[val_df["align_sim"] >= THRESHOLD].reset_index(drop=True)
print(f"\n정제: {len(val_df)} → {len(clean)}쌍 (임계값 {THRESHOLD:.3f})")

clean.to_csv(f"{OUT_DIR}/my_eval_clean.csv", index=False, encoding="utf-8-sig")
print(f"저장: {OUT_DIR}/my_eval_clean.csv")

In [ ]:
import glob
print("=== my_language_adapter_run 안 ===")
for p in glob.glob("/content/drive/MyDrive/FinHOLLY/my_language_adapter_run/**/*", recursive=True):
    print(p)

In [ ]:
import torch, pandas as pd, sacrebleu
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from peft import PeftModel

BASE = "facebook/nllb-200-distilled-1.3B"
ADAPTER = "/content/drive/MyDrive/FinHOLLY/my_language_adapter_run/checkpoint-epoch2"
SRC_LANG, TGT_LANG = "kor_Hang", "mya_Mymr"
device = "cuda"

tokenizer = AutoTokenizer.from_pretrained(BASE)
base_model = AutoModelForSeq2SeqLM.from_pretrained(BASE, torch_dtype=torch.float16)
model = PeftModel.from_pretrained(base_model, ADAPTER).to(device).eval()
print("어댑터 로드 완료")

clean = pd.read_csv("/content/drive/MyDrive/FinHOLLY/my_language_adapter_run/my_eval_clean.csv", encoding="utf-8-sig")
eval_df = clean.sample(min(500, len(clean)), random_state=7).reset_index(drop=True)
ko_texts = eval_df["ko"].tolist()
refs = eval_df["my"].tolist()

@torch.no_grad()
def translate_batch(ko_texts, use_adapter, batch_size=16):
    tokenizer.src_lang = SRC_LANG
    bos = tokenizer.convert_tokens_to_ids(TGT_LANG)
    out = []
    for i in range(0, len(ko_texts), batch_size):
        b = ko_texts[i:i+batch_size]
        inp = tokenizer(b, return_tensors="pt", padding=True, truncation=True, max_length=128).to(device)
        if use_adapter:
            ids = model.generate(**inp, forced_bos_token_id=bos, max_length=128, num_beams=4)
        else:
            with model.disable_adapter():
                ids = model.generate(**inp, forced_bos_token_id=bos, max_length=128, num_beams=4)
        out.extend(tokenizer.batch_decode(ids, skip_special_tokens=True))
    return out

print("순정 평가...")
base_hyps = translate_batch(ko_texts, False)
base_chrf = sacrebleu.corpus_chrf(base_hyps, [refs]).score
print(f"순정 chrF: {base_chrf:.2f}")

print("어댑터 평가...")
adp_hyps = translate_batch(ko_texts, True)
adp_chrf = sacrebleu.corpus_chrf(adp_hyps, [refs]).score
print(f"어댑터 chrF: {adp_chrf:.2f}")

print(f"\n=== 깨끗한 평가셋(500개) 결과 ===")
print(f"chrF: {base_chrf:.2f} → {adp_chrf:.2f} ({adp_chrf-base_chrf:+.2f})")

In [ ]:
import pandas as pd, sacrebleu

fin = pd.read_csv("/content/drive/MyDrive/FinHOLLY/exaone_generated/translated_pairs.csv", encoding="utf-8-sig")

fin_my = fin[fin["lang"]=="mya_Mymr"].dropna(subset=["translation"]).copy()
fin_my = fin_my[fin_my["translation"].astype(str).str.strip()!=""].reset_index(drop=True)
print(f"금융 미얀마 문장: {len(fin_my)}개")

ko_texts = fin_my["ko_sentence"].tolist()
refs = fin_my["translation"].tolist()

print("순정 평가...")
base_hyps = translate_batch(ko_texts, False)
base_chrf = sacrebleu.corpus_chrf(base_hyps, [refs]).score
print(f"순정 chrF: {base_chrf:.2f}")

print("어댑터 평가...")
adp_hyps = translate_batch(ko_texts, True)
adp_chrf = sacrebleu.corpus_chrf(adp_hyps, [refs]).score
print(f"어댑터 chrF: {adp_chrf:.2f}")

print(f"\n=== 금융 문장 평가 ===")
print(f"chrF: {base_chrf:.2f} → {adp_chrf:.2f} ({adp_chrf-base_chrf:+.2f})")

print("\n=== 샘플 ===")
for i in range(min(3, len(fin_my))):
    print(f"[한국어] {ko_texts[i]}")
    print(f"[정답]   {refs[i]}")
    print(f"[순정]   {base_hyps[i]}")
    print(f"[어댑터] {adp_hyps[i]}")
    print("---")

In [ ]:
import torch, pandas as pd, sacrebleu, gc
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from peft import PeftModel

BASE = "facebook/nllb-200-distilled-1.3B"
FIN = "/content/drive/MyDrive/FinHOLLY/domain_adapter_run/domain_adapter_final"
LANG = "/content/drive/MyDrive/FinHOLLY/my_language_adapter_run/checkpoint-epoch2"
SRC, TGT = "kor_Hang", "mya_Mymr"
device = "cuda"

tokenizer = AutoTokenizer.from_pretrained(BASE)
base = AutoModelForSeq2SeqLM.from_pretrained(BASE, torch_dtype=torch.float16)
model = PeftModel.from_pretrained(base, FIN, adapter_name="fin")
model.load_adapter(LANG, adapter_name="lang")
model.add_weighted_adapter(["fin","lang"], [1.0,1.0], "combo", combination_type="linear")
model = model.to(device).eval()
gc.collect(); torch.cuda.empty_cache()
print("로드 완료")

fin = pd.read_csv("/content/drive/MyDrive/FinHOLLY/exaone_generated/translated_pairs.csv", encoding="utf-8-sig")
fm = fin[fin["lang"]=="mya_Mymr"].dropna(subset=["translation"])
fm = fm[fm["translation"].astype(str).str.strip()!=""].reset_index(drop=True)
ko = fm["ko_sentence"].tolist(); refs = fm["translation"].tolist()
print(f"{len(ko)}개")

@torch.no_grad()
def tr(adapter):
    tokenizer.src_lang = SRC
    bos = tokenizer.convert_tokens_to_ids(TGT)
    if adapter: model.set_adapter(adapter)
    out=[]
    for i in range(0,len(ko),8):
        b=ko[i:i+8]
        inp=tokenizer(b,return_tensors="pt",padding=True,truncation=True,max_length=128).to(device)
        if adapter:
            ids=model.generate(**inp,forced_bos_token_id=bos,max_length=128,num_beams=4)
        else:
            with model.disable_adapter():
                ids=model.generate(**inp,forced_bos_token_id=bos,max_length=128,num_beams=4)
        out.extend(tokenizer.batch_decode(ids,skip_special_tokens=True))
    return out

for name,ad in [("A순정",None),("B금융","fin"),("C언어","lang"),("D조합","combo")]:
    h=tr(ad)
    print(f"{name}: {sacrebleu.corpus_chrf(h,[refs]).score:.2f}")
    if name=="D조합":
        print(f"\n[한국어] {ko[0]}\n[정답] {refs[0]}\n[조합] {h[0]}")

In [ ]:
!pip install -q groq

In [ ]:
import os, time, pandas as pd
from groq import Groq
from google.colab import userdata

os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')
client = Groq(api_key=os.environ["GROQ_API_KEY"])

heldout = pd.read_csv("/content/drive/MyDrive/FinHOLLY/data/my_eval_heldout_ko.csv", encoding="utf-8-sig")
print(f"held-out: {len(heldout)}개")

def translate_my(ko):
    prompt = f"""Translate this Korean financial sentence into Burmese (Myanmar).
Output ONLY the Burmese translation, no explanation.

Korean: {ko}"""
    try:
        r = client.chat.completions.create(
            model="openai/gpt-oss-120b",
            messages=[{"role":"user","content":prompt}],
            temperature=0.3, max_tokens=256)
        return r.choices[0].message.content.strip()
    except Exception as e:
        return f"[ERROR:{e}]"

refs = []
for i, ko in enumerate(heldout["ko_sentence"]):
    refs.append(translate_my(ko))
    if (i+1) % 20 == 0: print(f"  {i+1}/100")
    time.sleep(0.5)

heldout["my_ref"] = refs
heldout = heldout[~heldout["my_ref"].str.startswith("[ERROR", na=False)]
heldout.to_csv("/content/drive/MyDrive/FinHOLLY/data/my_eval_heldout_full.csv", index=False, encoding="utf-8-sig")
print(f"정답 생성 완료: {len(heldout)}개 (에러 제외)")

ev = pd.read_csv("/content/drive/MyDrive/FinHOLLY/data/my_eval_heldout_full.csv", encoding="utf-8-sig")
ev = ev.dropna(subset=["ko_sentence", "my_ref"])
ev = ev[ev["my_ref"].astype(str).str.strip() != ""]
ev = ev[~ev["my_ref"].astype(str).str.startswith("[ERROR")]
ev = ev.reset_index(drop=True)
ko = ev["ko_sentence"].astype(str).tolist()
refs = ev["my_ref"].astype(str).tolist()
print(f"평가(정제 후): {len(ko)}개")

results = {}
for name, ad in [("A순정",None),("B금융","fin"),("C언어","lang"),("D조합","combo")]:
    h = tr(ad)
    score = sacrebleu.corpus_chrf(h, [refs]).score
    results[name] = (score, h)
    print(f"{name}: chrF {score:.2f}")

print("\n=== 샘플 3개 ===")
for i in range(3):
    print(f"[한국어] {ko[i]}")
    print(f"[정답]   {refs[i]}")
    print(f"[B금융]  {results['B금융'][1][i]}")
    print(f"[D조합]  {results['D조합'][1][i]}")
    print("---")print(heldout[["ko_sentence","my_ref"]].head(3).to_string())

In [ ]:
ev = pd.read_csv("/content/drive/MyDrive/FinHOLLY/data/my_eval_heldout_full.csv", encoding="utf-8-sig")
ev = ev.dropna(subset=["ko_sentence", "my_ref"])
ev = ev[ev["my_ref"].astype(str).str.strip() != ""]
ev = ev[~ev["my_ref"].astype(str).str.startswith("[ERROR")]
ev = ev.reset_index(drop=True)
ko = ev["ko_sentence"].astype(str).tolist()
refs = ev["my_ref"].astype(str).tolist()
print(f"평가(정제 후): {len(ko)}개")

results = {}
for name, ad in [("A순정",None),("B금융","fin"),("C언어","lang"),("D조합","combo")]:
    h = tr(ad)
    score = sacrebleu.corpus_chrf(h, [refs]).score
    results[name] = (score, h)
    print(f"{name}: chrF {score:.2f}")

print("\n=== 샘플 3개 ===")
for i in range(3):
    print(f"[한국어] {ko[i]}")
    print(f"[정답]   {refs[i]}")
    print(f"[B금융]  {results['B금융'][1][i]}")
    print(f"[D조합]  {results['D조합'][1][i]}")
    print("---")

import torch, sacrebleu, gc
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from peft import PeftModel

BASE = "facebook/nllb-200-distilled-1.3B"
FIN = "/content/drive/MyDrive/FinHOLLY/domain_adapter_run/domain_adapter_final"
LANG = "/content/drive/MyDrive/FinHOLLY/my_language_adapter_run/checkpoint-epoch2"
SRC, TGT = "kor_Hang", "mya_Mymr"
device = "cuda"

tokenizer = AutoTokenizer.from_pretrained(BASE)
base = AutoModelForSeq2SeqLM.from_pretrained(BASE, torch_dtype=torch.float16)
model = PeftModel.from_pretrained(base, FIN, adapter_name="fin")
model.load_adapter(LANG, adapter_name="lang")
model.add_weighted_adapter(["fin","lang"], [1.0,1.0], "combo", combination_type="linear")
model = model.to(device).eval()
gc.collect(); torch.cuda.empty_cache()
print("로드 완료")

ev = pd.read_csv("/content/drive/MyDrive/FinHOLLY/data/my_eval_heldout_full.csv", encoding="utf-8-sig")
ko = ev["ko_sentence"].tolist(); refs = ev["my_ref"].tolist()
print(f"평가: {len(ko)}개")

@torch.no_grad()
def tr(adapter):
    tokenizer.src_lang = SRC
    bos = tokenizer.convert_tokens_to_ids(TGT)
    if adapter: model.set_adapter(adapter)
    out=[]
    for i in range(0,len(ko),8):
        b=ko[i:i+8]
        inp=tokenizer(b,return_tensors="pt",padding=True,truncation=True,max_length=128).to(device)
        if adapter:
            ids=model.generate(**inp,forced_bos_token_id=bos,max_length=128,num_beams=4)
        else:
            with model.disable_adapter():
                ids=model.generate(**inp,forced_bos_token_id=bos,max_length=128,num_beams=4)
        out.extend(tokenizer.batch_decode(ids,skip_special_tokens=True))
    return out

for name,ad in [("A순정",None),("B금융","fin"),("C언어","lang"),("D조합","combo")]:
    h=tr(ad)
    print(f"{name}: chrF {sacrebleu.corpus_chrf(h,[refs]).score:.2f}")

In [ ]:
import os, time, pandas as pd
from groq import Groq
from google.colab import userdata

os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')
client = Groq(api_key=os.environ["GROQ_API_KEY"])

heldout = pd.read_csv("/content/drive/MyDrive/FinHOLLY/data/my_eval_heldout_ko.csv", encoding="utf-8-sig")
ko_list = heldout["ko_sentence"].astype(str).tolist()

LANGS = {"vie_Latn":"Vietnamese", "ind_Latn":"Indonesian", "tha_Thai":"Thai", "tgl_Latn":"Filipino (Tagalog)"}

def translate(ko, lang_name):
    try:
        r = client.chat.completions.create(
            model="openai/gpt-oss-120b",
            messages=[{"role":"user","content":f"Translate this Korean financial sentence into {lang_name}. Output ONLY the translation.\n\nKorean: {ko}"}],
            temperature=0.3, max_tokens=256)
        return r.choices[0].message.content.strip()
    except Exception as e:
        return f"[ERROR:{e}]"

for code, name in LANGS.items():
    print(f"=== {name} 번역 중 ===")
    refs = []
    for i, ko in enumerate(ko_list):
        refs.append(translate(ko, name))
        if (i+1)%25==0: print(f"  {i+1}/100")
        time.sleep(0.4)
    heldout[f"ref_{code}"] = refs

heldout.to_csv("/content/drive/MyDrive/FinHOLLY/data/heldout_4lang_refs.csv", index=False, encoding="utf-8-sig")
print("4개 언어 정답 생성 완료")

In [ ]:
import torch, sacrebleu, gc, pandas as pd
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from peft import PeftModel

BASE = "facebook/nllb-200-distilled-1.3B"
FIN = "/content/drive/MyDrive/FinHOLLY/domain_adapter_run/domain_adapter_final"
SRC = "kor_Hang"
device = "cuda"

tokenizer = AutoTokenizer.from_pretrained(BASE)
base = AutoModelForSeq2SeqLM.from_pretrained(BASE, torch_dtype=torch.float16)
model = PeftModel.from_pretrained(base, FIN, adapter_name="fin").to(device).eval()
gc.collect(); torch.cuda.empty_cache()
print("금융 어댑터 로드 완료")

ev = pd.read_csv("/content/drive/MyDrive/FinHOLLY/data/heldout_4lang_refs.csv", encoding="utf-8-sig")
ko_all = ev["ko_sentence"].astype(str).tolist()

LANGS = {"vie_Latn":"베트남", "ind_Latn":"인니", "tha_Thai":"태국", "tgl_Latn":"필리핀"}

@torch.no_grad()
def tr(ko_texts, tgt, use_adapter):
    tokenizer.src_lang = SRC
    bos = tokenizer.convert_tokens_to_ids(tgt)
    if use_adapter: model.set_adapter("fin")
    out=[]
    for i in range(0,len(ko_texts),8):
        b=ko_texts[i:i+8]
        inp=tokenizer(b,return_tensors="pt",padding=True,truncation=True,max_length=128).to(device)
        if use_adapter:
            ids=model.generate(**inp,forced_bos_token_id=bos,max_length=128,num_beams=4)
        else:
            with model.disable_adapter():
                ids=model.generate(**inp,forced_bos_token_id=bos,max_length=128,num_beams=4)
        out.extend(tokenizer.batch_decode(ids,skip_special_tokens=True))
    return out

print("\n=== 언어별 순정 vs 금융 어댑터 (held-out 금융 문장) ===")
for code, kname in LANGS.items():

    sub = ev.dropna(subset=[f"ref_{code}"]).copy()
    sub = sub[~sub[f"ref_{code}"].astype(str).str.startswith("[ERROR")]
    sub = sub[sub[f"ref_{code}"].astype(str).str.strip()!=""]
    ko = sub["ko_sentence"].astype(str).tolist()
    refs = sub[f"ref_{code}"].astype(str).tolist()

    base_h = tr(ko, code, False)
    adp_h = tr(ko, code, True)
    base_s = sacrebleu.corpus_chrf(base_h, [refs]).score
    adp_s = sacrebleu.corpus_chrf(adp_h, [refs]).score
    print(f"{kname:4} ({len(ko)}개): 순정 {base_s:.2f} → 금융 {adp_s:.2f}  ({adp_s-base_s:+.2f})")

In [ ]:
!pip install -q transformers peft accelerate sentencepiece sacremoses

In [ ]:
import torch, gc
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from peft import PeftModel

BASE = "facebook/nllb-200-distilled-1.3B"
FIN = "/content/drive/MyDrive/FinHOLLY/domain_adapter_run/domain_adapter_final"
MERGED_DIR = "/content/drive/MyDrive/FinHOLLY/finholly_merged_hf"

print("모델 로드...")
tokenizer = AutoTokenizer.from_pretrained(BASE)
base = AutoModelForSeq2SeqLM.from_pretrained(BASE, torch_dtype=torch.float32, low_cpu_mem_usage=True)
model = PeftModel.from_pretrained(base, FIN)
print("병합...")
merged = model.merge_and_unload()
print("저장...")
merged.save_pretrained(MERGED_DIR, safe_serialization=True)
tokenizer.save_pretrained(MERGED_DIR)
print(f"완료: {MERGED_DIR}")

In [ ]:
!pip install -q ctranslate2

In [ ]:
MERGED_DIR = "/content/drive/MyDrive/FinHOLLY/finholly_merged_hf"
CT2_DIR = "/content/drive/MyDrive/FinHOLLY/finholly_ct2_int8"

!ct2-transformers-converter --model {MERGED_DIR} --output_dir {CT2_DIR} --quantization int8 --force
print("변환 완료")

In [ ]:
import glob, os
files = glob.glob(f"{CT2_DIR}/*")
print("생성된 파일:", files)

total = sum(os.path.getsize(f) for f in files if os.path.isfile(f))
print(f"총 크기: {total/1e9:.2f} GB")

In [ ]:
!pip install -q sentencepiece

In [ ]:
import ctranslate2
from transformers import AutoTokenizer

CT2_DIR = "/content/drive/MyDrive/FinHOLLY/finholly_ct2_int8"
MERGED_DIR = "/content/drive/MyDrive/FinHOLLY/finholly_merged_hf"

translator = ctranslate2.Translator(CT2_DIR, device="cpu")
tokenizer = AutoTokenizer.from_pretrained(MERGED_DIR)

def translate(ko, tgt_lang):
    tokenizer.src_lang = "kor_Hang"
    src = tokenizer.convert_ids_to_tokens(tokenizer.encode(ko))
    results = translator.translate_batch([src], target_prefix=[[tgt_lang]])
    tokens = results[0].hypotheses[0]
    if tokens and tokens[0] == tgt_lang:
        tokens = tokens[1:]
    return tokenizer.decode(tokenizer.convert_tokens_to_ids(tokens))

test = "가산금리가 적용되어 이자가 올랐습니다"
print("한국어:", test)
for lang, name in [("mya_Mymr","미얀마"), ("vie_Latn","베트남"), ("ind_Latn","인니"), ("tha_Thai","태국"), ("tgl_Latn","필리핀")]:
    print(f"[{name}] {translate(test, lang)}")

In [ ]:
!pip install -q ctranslate2 transformers sentencepiece sacrebleu datasets

In [ ]:
!wget -q https://dl.fbaipublicfiles.com/nllb/flores200_dataset.tar.gz -O flores200.tar.gz
!tar xzf flores200.tar.gz
!echo "=== devtest 폴더 파일 (우리 언어) ==="
!ls flores200_dataset/devtest/ | grep -E "kor|vie|ind|tha|tgl|mya"

In [ ]:
import ctranslate2
from transformers import AutoTokenizer
import sacrebleu

CT2_DIR = "/content/drive/MyDrive/FinHOLLY/finholly_ct2_int8"
TOK_DIR = "/content/drive/MyDrive/FinHOLLY/finholly_merged_hf"
FLORES = "flores200_dataset/devtest"

translator = ctranslate2.Translator(CT2_DIR, device="cuda")
tokenizer = AutoTokenizer.from_pretrained(TOK_DIR)

with open(f"{FLORES}/kor_Hang.devtest", encoding="utf-8") as f:
    ko_lines = [l.strip() for l in f]
print(f"한국어 소스: {len(ko_lines)}문장")

def translate_all(ko_texts, tgt_lang, batch_size=16):
    tokenizer.src_lang = "kor_Hang"
    out = []
    for i in range(0, len(ko_texts), batch_size):
        batch = ko_texts[i:i+batch_size]
        srcs = [tokenizer.convert_ids_to_tokens(tokenizer.encode(t)) for t in batch]
        results = translator.translate_batch(srcs, target_prefix=[[tgt_lang]]*len(srcs))
        for r in results:
            toks = r.hypotheses[0]
            if toks and toks[0]==tgt_lang: toks=toks[1:]
            out.append(tokenizer.decode(tokenizer.convert_tokens_to_ids(toks)))
        if (i+16) % 160 == 0: print(f"  {i+16}/{len(ko_texts)}")
    return out

LANGS = {"vie_Latn":"베트남", "ind_Latn":"인니", "tha_Thai":"태국", "tgl_Latn":"필리핀", "mya_Mymr":"미얀마"}

print("\n=== FLORES-200 기준 우리 모델 (금융 어댑터) ===")
for code, name in LANGS.items():
    with open(f"{FLORES}/{code}.devtest", encoding="utf-8") as f:
        refs = [l.strip() for l in f]
    hyps = translate_all(ko_lines, code)
    chrf = sacrebleu.corpus_chrf(hyps, [refs]).score
    bleu = sacrebleu.corpus_bleu(hyps, [refs]).score
    print(f"{name}: chrF {chrf:.2f} | spBLEU {bleu:.2f}")